# Streaming Progress for Long-Running Jobs — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/system-design/11-1/streaming-progress-long-running-jobs.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/system-design/11-1/streaming-progress-long-running-jobs.ipynb)

Companion to: [Streaming Progress for Long-Running Jobs](https://anacodicai.org/learn/system-design/11-1/streaming-progress-long-running-jobs)


## What you'll build

1. A generic "start/done" event stream — and see why a client can't tell
   which phase is running.
2. Named phase events plus a distinct **interrupt** event type for a
   human-review pause.
3. A **durable, replayable log**: simulate a client disconnecting mid-run and
   recovering everything it missed on reconnection.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
print("ready — standard library only")

ready — standard library only


## Step 1 — the durable log

Every event is appended here BEFORE being pushed to any live listener, so a
reconnecting client can ask "what did I miss since index N" instead of losing
whatever happened while it was disconnected.

In [2]:
class DurableEventLog:
    def __init__(self):
        self.events = []

    def append(self, event):
        self.events.append(event)
        return len(self.events) - 1

    def since(self, last_seen_index):
        return self.events[last_seen_index + 1:]

print("DurableEventLog ready")

DurableEventLog ready


## Step 2 — generic events (layer 1 only)

Look at the raw event list below — there's no way to tell which of the three
phases the second `start`/`done` pair belongs to without separately counting.

In [3]:
def run_pipeline_generic(log):
    for _ in ["ingest", "grade", "persist"]:
        log.append({"type": "start"})
        log.append({"type": "done"})

generic_log = DurableEventLog()
run_pipeline_generic(generic_log)
for e in generic_log.events:
    print(" ", e)

  {'type': 'start'}
  {'type': 'done'}
  {'type': 'start'}
  {'type': 'done'}
  {'type': 'start'}
  {'type': 'done'}


## Step 3 — named events + a distinct interrupt type

**Try it:** add a 4th phase (say `"notify"`) after `persist` and re-run —
notice you don't have to change `render_phase_timeline` at all; it reads
whatever phase names show up in the events.

In [4]:
def run_pipeline_named_with_interrupt(log):
    log.append({"type": "ingest_start"})
    log.append({"type": "ingest_done"})
    log.append({"type": "grade_start"})
    log.append({
        "type": "interrupt",
        "gate": "low_confidence_review",
        "message": "Confidence below threshold; awaiting human decision.",
    })
    log.append({"type": "grade_resumed", "decision": "override_approved"})
    log.append({"type": "grade_done"})
    log.append({"type": "persist_start"})
    log.append({"type": "persist_done"})

def render_phase_timeline(events):
    timeline = []
    for e in events:
        t = e["type"]
        if t.endswith("_start"):
            timeline.append(f"{t.removesuffix('_start')}: started")
        elif t.endswith("_done"):
            timeline.append(f"{t.removesuffix('_done')}: finished")
        elif t == "interrupt":
            timeline.append(f"PAUSED -- {e['gate']}: {e['message']}")
        elif t.endswith("_resumed"):
            timeline.append(f"resumed: {e['decision']}")
    return timeline

named_log = DurableEventLog()
run_pipeline_named_with_interrupt(named_log)
for line in render_phase_timeline(named_log.events):
    print(" ", line)

  ingest: started
  ingest: finished
  grade: started
  PAUSED -- low_confidence_review: Confidence below threshold; awaiting human decision.
  resumed: override_approved
  grade: finished
  persist: started
  persist: finished


## Step 4 — reconnection: replay, not loss

A client disconnects after event index 2 and reconnects later. It asks the
log for everything since index 2.

In [5]:
client_last_seen = 2
print(f"client's last confirmed event index: {client_last_seen}")
missed = named_log.since(client_last_seen)
print(f"events the client missed while disconnected: {len(missed)}")
for e in missed:
    print(" ", e)

client's last confirmed event index: 2
events the client missed while disconnected: 5
  {'type': 'interrupt', 'gate': 'low_confidence_review', 'message': 'Confidence below threshold; awaiting human decision.'}
  {'type': 'grade_resumed', 'decision': 'override_approved'}
  {'type': 'grade_done'}
  {'type': 'persist_start'}
  {'type': 'persist_done'}


## Step 5 — prove the replay recovers the interrupt

The one event a client absolutely cannot afford to silently miss is the
interrupt — it means a human needs to act, and there's no other signal
telling them that.

In [6]:
assert len(missed) == len(named_log.events) - (client_last_seen + 1)
replayed_timeline = render_phase_timeline(missed)
assert any("PAUSED" in line for line in replayed_timeline)
print("assert passed: replay from the durable log recovers the interrupt "
      "event the client would otherwise have silently missed")

assert passed: replay from the durable log recovers the interrupt event the client would otherwise have silently missed


## Try it yourself

- Change `client_last_seen` to `0` or `5` and re-run Step 4/5 — the missed
  count changes exactly with it (`len(events) - (last_seen + 1)`).
- Delete the `log.append(...)` call before pushing an event "live" (imagine
  a live-only stream with no durable log) and think through what a
  reconnecting client would see instead: nothing about the interrupt, ever.